# Chapter 7 — Cost, Latency and Budgets

*Real agents fail on cost before they fail on safety.*

## Objective

**Why.** Agents fail on cost before they fail on safety — a single misconfigured loop can bill thousands before anyone notices.

**What.** A budget is a first-class *system* constraint across four axes (tokens, seconds, tool calls, dollars), enforced by the loop and the model adapter — never by the agent itself.

**How.** Run a task under three budget profiles against a real environment, watch the loop degrade, then measure a real model's tokens, latency, and cost so the budget is enforceable end-to-end.

In [ ]:
from pydantic import BaseModel
from glassloop.core import (
    AgentState, BaseAgent, Budget, BudgetTracker, Finish, TaskSpec, ToolCall, run_loop,
)
from proofloop.evaluation import collect, summarize
from glassloop.tools import GovernedToolExecutor, RiskLevel, Tool, ToolRegistry

## A `Budget` caps each axis

Tokens, seconds, tool calls, dollars. `None` on an axis means unlimited. The `BudgetTracker` accumulates `Consumption` and reports the first axis that hits its cap.

In [ ]:
class PingAgent(BaseAgent):
    def propose_action(self, state):
        if state.step >= 5:
            return Finish(output='done')
        return ToolCall(tool_name='ping', arguments={})

# A real environment: the action runs through the Chapter 6 executor and a real tool.
class PingIn(BaseModel): pass
class PingOut(BaseModel): pong: bool

registry = ToolRegistry()
registry.register(Tool(name='ping', description='a no-op tool',
                       input_schema=PingIn, output_schema=PingOut,
                       risk=RiskLevel.LOW, fn=lambda: {'pong': True}))

class ToolEnv:
    def __init__(self, executor): self._ex = executor
    def step(self, action):
        return {'success': self._ex.execute(action).success}

env = ToolEnv(GovernedToolExecutor(registry))
task = TaskSpec(goal='do 5 pings')

def run_with(budget: Budget) -> dict:
    tracker = BudgetTracker(budget)
    traj = collect(task, run_loop(PingAgent(), env, AgentState(task=task),
                                  max_steps=20, budget_tracker=tracker))
    s = summarize(traj)
    s['budget_reason'] = tracker.reason_exhausted()
    return s

for name, b in [('generous', Budget(tool_calls=100)),
                ('tight',    Budget(tool_calls=5)),
                ('hostile',  Budget(tool_calls=1))]:
    s = run_with(b)
    print(f'{name:>10}: status={s["status"]:<10} tool_calls={s["tool_calls"]:<2} reason={s["budget_reason"]!r}')

Under a generous budget the loop terminates on `Finish`. Under a hostile budget the loop synthesizes an `Escalate` record with `source=budget` and marks the state as `failed`. Either way the trajectory is inspectable.

## Model adapters report their own usage

`QwenAdapter` reports `last_input_tokens` and `last_output_tokens` after each call — the real counts, not an estimate — and it runs a real model, so the latency you measure is the latency you'll pay. The dollars axis is then token count times your deployment's rate.

In [ ]:
import time
from glassloop.models import QwenAdapter

lm = QwenAdapter()
lm.complete('warm up')                  # the first call loads the model
start = time.time()
out = lm.complete('Summarize the overdraft fee policy in one sentence.')
elapsed = time.time() - start
print(f'latency: {elapsed:.2f}s')
print(f'last_input_tokens:  {lm.last_input_tokens}')
print(f'last_output_tokens: {lm.last_output_tokens}')

In [ ]:
# The dollars axis: real measured tokens times a rate you set.
price_per_1k_tokens = 0.0005     # your deployment's rate, in USD
tokens = lm.last_input_tokens + lm.last_output_tokens
print(f'tokens: {tokens}')
print(f'cost at ${price_per_1k_tokens}/1k tokens: ${tokens / 1000 * price_per_1k_tokens:.6f}')

## Anti-patterns flagged here

- Counting tokens only on the model and forgetting tool latency.
- Per-call retries without a global budget.
- Caching keys that include timestamps.

In [ ]:
# Self-check
assert run_with(Budget(tool_calls=1))['status'] == 'failed'
assert run_with(Budget(tool_calls=100))['status'] == 'done'
print('OK')

## Exercises

Worked solutions follow — the three budget axes (calls, cost, the synthetic failure record) made concrete. Exercise 7.2 reuses the `QwenAdapter` loaded above.

In [ ]:
# Exercise 7.1 — why a sufficient budget can still fail
print('tool_calls=5:', run_with(Budget(tool_calls=5))['status'])
print('tool_calls=6:', run_with(Budget(tool_calls=6))['status'])
# Five pings exhaust a budget of five before the Finish step runs (the tracker is
# checked at the top of each iteration); six leaves room for the final step.

In [ ]:
# Exercise 7.2 — the real cost of generation
RATE = 0.0005   # USD per 1k tokens
for label, prompt in [('short', 'Reply with the single word: ok.'),
                      ('long',  'Explain the overdraft fee dispute process in detail.')]:
    lm.complete(prompt)
    t = lm.last_input_tokens + lm.last_output_tokens
    print(f'  {label:<5} tokens={t:<4} cost=${t / 1000 * RATE:.6f}')

In [ ]:
# Exercise 7.3 — the synthetic budget record
tracker = BudgetTracker(Budget(tool_calls=1))
traj = collect(task, run_loop(PingAgent(), env, AgentState(task=task),
                              max_steps=20, budget_tracker=tracker))
last = traj.records[-1]
print('final action kind:', last.action.kind)
print('context:', getattr(last.action, 'context', None))
# Same record shape as any terminal step — downstream evaluation needs no special case.